# 🔍 Detecção de Anomalias em Transações com Python

Pipeline completo de Machine Learning para detecção de fraude em cartões de crédito.

Aborda desde a análise exploratória até modelos avançados com explicabilidade via SHAP.

**Dataset:** [Credit Card Fraud Detection – Kaggle / TensorFlow](https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv)

---
## Índice
1. [Importação e Carregamento](#1)
2. [Análise Exploratória (EDA)](#2)
3. [Tratamento de Dados](#3)
4. [Feature Engineering](#4)
5. [Split Correto + SMOTE no Treino](#5)
6. [Modelos: LR / RF / XGBoost](#6)
7. [Validação Cruzada com StratifiedKFold](#7)
8. [GridSearch Aprimorado](#8)
9. [Comparativo Final dos Modelos](#9)
10. [Explicabilidade com SHAP](#10)

## 1. Importação e Carregamento dos Dados <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")

url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(url)

print(f"Shape: {df.shape}")
df.head()

## 2. Análise Exploratória de Dados (EDA) <a id='2'></a>

Antes de modelar, precisamos entender a distribuição dos dados, identificar padrões
e compreender o grau de desbalanceamento das classes.

In [ ]:
# 2.1 Distribuição das classes
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["Normal (0)", "Fraude (1)"], class_counts.values, color=["steelblue", "tomato"])
axes[0].set_title("Contagem de Transações por Classe")
axes[0].set_ylabel("Quantidade")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, f"{v:,}", ha="center", fontweight="bold")

axes[1].pie(class_counts.values, labels=["Normal (0)", "Fraude (1)"],
            autopct="%1.2f%%", colors=["steelblue", "tomato"], startangle=90)
axes[1].set_title("Proporção das Classes")

plt.suptitle("Desbalanceamento do Dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Transações normais : {class_counts[0]:,} ({class_pct[0]:.2f}%)")
print(f"Transações fraude  : {class_counts[1]:,} ({class_pct[1]:.2f}%)")
print(f"Razão de desbalanceamento: {class_counts[0]/class_counts[1]:.0f}:1")

In [ ]:
# 2.2 Distribuição do Amount por classe
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for cls, label, color, ax in zip([0, 1], ["Normal", "Fraude"], ["steelblue", "tomato"], axes):
    subset = df[df["Class"] == cls]["Amount"]
    ax.hist(subset, bins=60, color=color, alpha=0.8, edgecolor="white")
    ax.set_title(f"Distribuição do Amount — {label}")
    ax.set_xlabel("Amount (R$)")
    ax.set_ylabel("Frequência")
    ax.axvline(subset.median(), color="black", linestyle="--", label=f"Mediana: {subset.median():.2f}")
    ax.legend()

plt.suptitle("Amount por Classe", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 Distribuição do Time
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for cls, label, color, ax in zip([0, 1], ["Normal", "Fraude"], ["steelblue", "tomato"], axes):
    subset = df[df["Class"] == cls]["Time"]
    ax.hist(subset, bins=50, color=color, alpha=0.8, edgecolor="white")
    ax.set_title(f"Distribuição do Time — {label}")
    ax.set_xlabel("Tempo (segundos desde 1ª transação)")
    ax.set_ylabel("Frequência")

plt.suptitle("Time por Classe", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 2.4 Boxplots: Amount e Time por classe
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.boxplot(column="Amount", by="Class", ax=axes[0], 
           patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.6))
axes[0].set_title("Amount por Classe")
axes[0].set_xlabel("Classe (0=Normal, 1=Fraude)")
axes[0].set_ylabel("Amount")
plt.sca(axes[0])
plt.title("Amount por Classe")

df.boxplot(column="Time", by="Class", ax=axes[1],
           patch_artist=True,
           boxprops=dict(facecolor="tomato", alpha=0.6))
axes[1].set_title("Time por Classe")
axes[1].set_xlabel("Classe (0=Normal, 1=Fraude)")
plt.sca(axes[1])
plt.title("Time por Classe")

plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# 2.5 Heatmap de correlação (features V1–V10 + Amount + Time)
cols_sample = [c for c in df.columns if c.startswith("V")][:10] + ["Amount", "Time", "Class"]

plt.figure(figsize=(14, 10))
corr = df[cols_sample].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={"size": 8})

plt.title("Matriz de Correlação (V1–V10 + Amount + Time + Class)", 
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 2.6 Distribuição das features V1–V5 por classe
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate([f"V{j}" for j in range(1, 11)]):
    for cls, label, color in zip([0, 1], ["Normal", "Fraude"], ["steelblue", "tomato"]):
        subset = df[df["Class"] == cls][col]
        axes[i].hist(subset, bins=50, alpha=0.5, color=color, label=label, density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

plt.suptitle("Distribuição das Features V1–V10 por Classe", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Tratamento de Dados <a id='3'></a>

Verificação de valores nulos, duplicatas e estatísticas descritivas antes de qualquer modelagem.

In [ ]:
# 3.1 Informações gerais
print("=" * 50)
print("INFO DO DATAFRAME")
print("=" * 50)
df.info()

In [ ]:
# 3.2 Estatísticas descritivas
print("\nESTATÍSTICAS DESCRITIVAS")
print("=" * 50)
df.describe().T.style.background_gradient(cmap="Blues")

In [ ]:
# 3.3 Valores nulos e duplicatas
print("VALORES NULOS POR COLUNA:")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "✅ Nenhum valor nulo encontrado.")

print(f"\nDUPLICATAS: {df.duplicated().sum()} linhas duplicadas")

# Remove duplicatas, se houver
if df.duplicated().sum() > 0:
    df = df.drop_duplicates()
    print(f"  ↳ Duplicatas removidas. Novo shape: {df.shape}")
else:
    print("✅ Nenhuma duplicata encontrada.")

## 4. Feature Engineering <a id='4'></a>

Criamos as features derivadas **antes** do split, mas sem aplicar o scaler ainda
(isso será feito dentro do Pipeline para evitar data leakage).

In [ ]:
import numpy as np

# Log do Amount para reduzir skewness
df["Amount_log"] = np.log1p(df["Amount"])

# Hora do dia (Time em segundos → ciclica)
df["Hour"] = (df["Time"] // 3600) % 24

# Dropar colunas originais que serão substituídas ou são redundantes
df_model = df.drop(columns=["Time", "Amount"])

print("Features finais para modelagem:")
print(df_model.columns.tolist())
print(f"\nShape: {df_model.shape}")

## 5. Train/Test Split + SMOTE Corretamente Aplicado <a id='5'></a>

> ⚠️ **Data leakage evitado**: o SMOTE é aplicado **somente nos dados de treino**, 
> após a separação. O conjunto de teste nunca influencia o balanceamento.
>
> ⚠️ **Scaler no Pipeline**: o `StandardScaler` ficará dentro do Pipeline de cada modelo,
> garantindo que o fit ocorra apenas sobre os dados de treino em cada fold.

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = df_model.drop("Class", axis=1)
y = df_model["Class"]

# 1) Split primeiro
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.3, random_state=42
)

print(f"Treino  → X: {X_train.shape} | Fraudes: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Teste   → X: {X_test.shape}  | Fraudes: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

# 2) SMOTE apenas no treino
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"\nApós SMOTE no treino:")
print(f"  Classe 0 (Normal): {(y_train_res == 0).sum():,}")
print(f"  Classe 1 (Fraude): {(y_train_res == 1).sum():,}")
print(f"  Shape final: {X_train_res.shape}")

## 6. Treinamento dos Modelos <a id='6'></a>

Todos os modelos usam **Pipeline com StandardScaler interno** para eliminar leakage no scaler.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                              precision_recall_curve, roc_curve,
                              average_precision_score)

# ── Dicionário de modelos ──────────────────────────────────────────────────────
modelos = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=100, max_depth=10,
            class_weight="balanced", n_jobs=-1, random_state=42
        ))
    ]),
    "XGBoost": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", XGBClassifier(
            scale_pos_weight=10, eval_metric="logloss",
            n_estimators=100, random_state=42, n_jobs=-1
        ))
    ])
}

resultados = {}

for nome, pipeline in modelos.items():
    pipeline.fit(X_train_res, y_train_res)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    resultados[nome] = {
        "pipeline": pipeline,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "auc_roc": roc_auc_score(y_test, y_prob),
        "avg_precision": average_precision_score(y_test, y_prob),
    }
    
    print(f"\n{'='*55}")
    print(f"  {nome}")
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred, target_names=["Normal", "Fraude"]))

In [ ]:
# Curvas ROC e Precision-Recall lado a lado para todos os modelos
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cores = ["steelblue", "darkorange", "seagreen"]

for (nome, res), cor in zip(resultados.items(), cores):
    # ROC
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    axes[0].plot(fpr, tpr, color=cor, lw=2,
                 label=f"{nome} (AUC={res['auc_roc']:.3f})")
    
    # PR
    prec, rec, _ = precision_recall_curve(y_test, res["y_prob"])
    axes[1].plot(rec, prec, color=cor, lw=2,
                 label=f"{nome} (AP={res['avg_precision']:.3f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title("ROC Curve — Comparativo", fontsize=12, fontweight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].axhline(y_test.mean(), color="red", linestyle="--", alpha=0.5, label="Baseline")
axes[1].set_title("Precision-Recall Curve — Comparativo", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Ajuste de threshold para o melhor modelo (XGBoost)
y_prob_xgb = resultados["XGBoost"]["y_prob"]

threshold = 0.3
y_pred_custom = (y_prob_xgb > threshold).astype(int)

print(f"XGBoost com threshold = {threshold}")
print(classification_report(y_test, y_pred_custom, target_names=["Normal", "Fraude"]))

## 7. Validação Cruzada com StratifiedKFold <a id='7'></a>

`StratifiedKFold` preserva a proporção de classes em cada fold — essencial para datasets desbalanceados.
Aqui aplicamos no conjunto completo de treino, com SMOTE em cada fold individualmente via `Pipeline + imblearn`.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_modelos = {
    "Logistic Regression": ImbPipeline([
        ("smote", SMOTE(random_state=42)),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": ImbPipeline([
        ("smote", SMOTE(random_state=42)),
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=100, max_depth=10,
                                        class_weight="balanced", n_jobs=-1, random_state=42))
    ]),
    "XGBoost": ImbPipeline([
        ("smote", SMOTE(random_state=42)),
        ("scaler", StandardScaler()),
        ("clf", XGBClassifier(scale_pos_weight=10, eval_metric="logloss",
                               n_estimators=100, random_state=42, n_jobs=-1))
    ]),
}

metricas_cv = ["recall", "precision", "f1", "roc_auc"]
print("Validação Cruzada (5-fold StratifiedKFold)\n")

for nome, pipe in cv_modelos.items():
    scores = cross_validate(pipe, X_train, y_train, cv=skf,
                            scoring=metricas_cv, n_jobs=-1)
    print(f"  {nome}")
    for m in metricas_cv:
        vals = scores[f"test_{m}"]
        print(f"    {m:12s}: {vals.mean():.4f} ± {vals.std():.4f}")
    print()

## 8. Ajuste de Hiperparâmetros — GridSearchCV Aprimorado <a id='8'></a>

Grid expandido para o XGBoost com os parâmetros mais impactantes para dados desbalanceados.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_xgb = {
    "clf__max_depth"         : [3, 5, 7],
    "clf__n_estimators"      : [100, 200],
    "clf__learning_rate"     : [0.05, 0.1],
    "clf__subsample"         : [0.8, 1.0],
    "clf__colsample_bytree"  : [0.8, 1.0],
    "clf__gamma"             : [0, 1],
    "clf__min_child_weight"  : [1, 5],
}

grid_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier(scale_pos_weight=10, eval_metric="logloss",
                           random_state=42, n_jobs=-1))
])

# Usar RandomizedSearchCV para grid grande (mais eficiente)
from sklearn.model_selection import RandomizedSearchCV

grid_search = RandomizedSearchCV(
    grid_pipeline,
    param_distributions=param_grid_xgb,
    n_iter=20,
    scoring="recall",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    n_jobs=-1,
    random_state=42,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n✅ Melhores hiperparâmetros encontrados:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")

print(f"\nRecall médio (CV): {grid_search.best_score_:.4f}")

# Avalia no test set
y_pred_grid = grid_search.best_estimator_.predict(X_test)
y_prob_grid = grid_search.best_estimator_.predict_proba(X_test)[:, 1]

print("\nResultado no Test Set:")
print(classification_report(y_test, y_pred_grid, target_names=["Normal", "Fraude"]))

## 9. Comparativo Final dos Modelos <a id='9'></a>

Tabela unificada com as métricas de todos os modelos para decisão fundamentada.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd

linhas = []

for nome, res in resultados.items():
    yp = res["y_pred"]
    linhas.append({
        "Modelo"           : nome,
        "AUC-ROC"          : round(res["auc_roc"], 4),
        "Avg Precision"    : round(res["avg_precision"], 4),
        "Recall (Fraude)"  : round(recall_score(y_test, yp, pos_label=1), 4),
        "Precision (Fraude)": round(precision_score(y_test, yp, pos_label=1), 4),
        "F1 (Fraude)"      : round(f1_score(y_test, yp, pos_label=1), 4),
    })

# Adiciona o melhor do GridSearch
yp_g = y_pred_grid
linhas.append({
    "Modelo"           : "XGBoost (GridSearch)",
    "AUC-ROC"          : round(roc_auc_score(y_test, y_prob_grid), 4),
    "Avg Precision"    : round(average_precision_score(y_test, y_prob_grid), 4),
    "Recall (Fraude)"  : round(recall_score(y_test, yp_g, pos_label=1), 4),
    "Precision (Fraude)": round(precision_score(y_test, yp_g, pos_label=1), 4),
    "F1 (Fraude)"      : round(f1_score(y_test, yp_g, pos_label=1), 4),
})

df_comp = pd.DataFrame(linhas).set_index("Modelo")
df_comp.style.highlight_max(axis=0, color="lightgreen").format("{:.4f}")

In [ ]:
# Visualização do comparativo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metricas_bar = ["AUC-ROC", "Recall (Fraude)", "F1 (Fraude)"]
cores_bar = ["steelblue", "darkorange", "seagreen", "mediumpurple"]

for ax, metrica in zip(axes, metricas_bar):
    vals = df_comp[metrica]
    bars = ax.barh(vals.index, vals.values, color=cores_bar[:len(vals)])
    ax.set_title(metrica, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.axvline(0.8, color="red", linestyle="--", alpha=0.4, label="0.8")
    for bar, v in zip(bars, vals.values):
        ax.text(v + 0.005, bar.get_y() + bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=9)

plt.suptitle("Comparativo de Modelos — Detecção de Fraude", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Explicabilidade com SHAP <a id='10'></a>

SHAP (SHapley Additive exPlanations) revela quais features mais influenciam cada predição.
Essencial para auditabilidade em modelos de detecção de fraude.

In [ ]:
import shap

# Extrai o classificador XGBoost do pipeline para o SHAP
xgb_clf = resultados["XGBoost"]["pipeline"].named_steps["clf"]
scaler_clf = resultados["XGBoost"]["pipeline"].named_steps["scaler"]

# Transforma o test set com o scaler do pipeline
X_test_scaled = scaler_clf.transform(X_test)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Calcula SHAP values (amostra para performance)
sample_size = 500
X_shap = X_test_scaled_df.sample(sample_size, random_state=42)

explainer = shap.Explainer(xgb_clf, X_shap)
shap_values = explainer(X_shap)

print(f"SHAP values calculados para {sample_size} amostras do test set.")

In [ ]:
# SHAP Bar Plot — Importância média global
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title("SHAP — Importância Global das Features (Top 15)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Beeswarm — Distribuição de impacto por feature
plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP Beeswarm — Impacto por Feature", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Waterfall — Explicação de uma transação individual (suspeita)
fraudes_idx = y_test[y_test == 1].index
sample_idx_local = list(X_shap.index).index(fraudes_idx[0]) if fraudes_idx[0] in X_shap.index else 0

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[sample_idx_local], show=False)
plt.title("SHAP Waterfall — Explicação de uma Transação Fraudulenta", fontweight="bold")
plt.tight_layout()
plt.show()

## Conclusões

| Aspecto | Resultado |
|---|---|
| Dataset | 284.807 transações, 0.17% de fraudes |
| Melhor modelo | XGBoost (com GridSearch) |
| Técnicas de balanceamento | SMOTE aplicado **apenas no treino** dentro de cada fold |
| Validação | StratifiedKFold 5-fold |
| Explicabilidade | SHAP (global + individual) |

**Lições-chave:**
- Accuracy é enganosa em datasets desbalanceados → priorize **Recall** e **AUC-PR**
- Data leakage em SMOTE e StandardScaler são os erros mais comuns — evitados aqui com Pipelines
- SHAP transforma o modelo de uma "caixa preta" em algo auditável e justificável